# Graph Isomorphism Network on REDDIT-MULTI-12K

This notebook trains a [Graph Isomorphism Network (GIN)](https://arxiv.org/abs/1810.00826) on the `REDDIT-MULTI-12K` dataset from the TU Dataset collection using [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/). It is designed to run smoothly on Google Colab (including the free GPU runtime), but it will also work in any environment with the required dependencies installed.

## 1. Environment preparation

Running on Google Colab? Execute the following cell to install the exact dependency versions tested with this notebook. If you are working locally and already have a suitable PyTorch + PyTorch Geometric stack, you can skip it. The commands automatically detect Colab and avoid modifying environments unnecessarily.

In [ ]:
import sys
import subprocess

if 'google.colab' in sys.modules:
    print('Running on Colab – installing PyTorch Geometric dependencies (this may take a minute).')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
        'torch==2.2.2+cu121', 'torchvision==0.17.2+cu121', 'torchaudio==2.2.2+cu121'
    ])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'torch_geometric',
        '-f', 'https://data.pyg.org/whl/torch-2.2.0+cu121.html'
    ])
else:
    print('Colab not detected. Make sure torch and torch_geometric are available in your environment.')


## 2. Imports and configuration

We configure the training device, random seeds, and a few hyperparameters. The notebook automatically uses a GPU when available.

In [ ]:
import os
import random
from dataclasses import dataclass

import torch
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool, BatchNorm

SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

@dataclass
class TrainingConfig:
    batch_size: int = 128
    hidden_channels: int = 128
    num_layers: int = 5
    dropout: float = 0.5
    lr: float = 5e-4
    weight_decay: float = 5e-5
    max_epochs: int = 80
    lr_patience: int = 10

config = TrainingConfig()
config


## 3. Dataset loading and exploration

`REDDIT-MULTI-12K` is a graph classification dataset containing Reddit discussion threads. Each graph describes a thread where nodes correspond to users and edges represent interactions. The `TUDataset` utility handles downloading and caching automatically.

In [ ]:
dataset = TUDataset(root='data/TUDataset', name='REDDIT-MULTI-12K')
print(dataset)
print(f'Number of classes: {dataset.num_classes}')
print(f'Number of node features: {dataset.num_features}')

# Inspect a single graph
sample = dataset[0]
print(sample)
print(f'Class label: {sample.y.item()}')
print(f'Nodes: {sample.num_nodes}, Edges: {sample.num_edges}')


## 4. Train/validation/test split

The TU datasets do not ship with a predefined split. We create an 80/10/10 stratified split so that the class distribution is balanced across training, validation, and testing subsets.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

labels = dataset.data.y.numpy()
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, temp_idx = next(sss.split(torch.zeros(len(labels)), labels))

labels_temp = labels[temp_idx]
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_idx, test_idx = next(sss_val.split(torch.zeros(len(labels_temp)), labels_temp))

val_idx = temp_idx[val_idx]
test_idx = temp_idx[test_idx]

train_dataset = dataset[train_idx]
val_dataset = dataset[val_idx]
test_dataset = dataset[test_idx]

print(f"Train graphs: {len(train_dataset)}")
print(f"Validation graphs: {len(val_dataset)}")
print(f"Test graphs: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size)


## 5. Model definition

We implement a GIN with learnable node embeddings through MLPs applied after message aggregation. Batch normalization and dropout are added to stabilize training and reduce overfitting.

In [ ]:
class GINNet(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, hidden_channels: int, num_layers: int, dropout: float):
        super().__init__()
        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.dropout = dropout

        for layer in range(num_layers):
            nn_layers = nn.Sequential(
                nn.Linear(in_channels if layer == 0 else hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            )
            conv = GINConv(nn_layers)
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(hidden_channels))

        self.readout = nn.Linear(hidden_channels, out_channels)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        if x is None:
            # Some TU datasets lack node attributes. Use one-hot encodings of node degrees instead.
            x = torch.ones((data.num_nodes, 1), device=edge_index.device)

        for conv, bn in zip(self.convs, self.batch_norms):
            x = conv(x, edge_index)
            x = bn(x)
            x = x.relu()
            x = nn.functional.dropout(x, p=self.dropout, training=self.training)

        x = global_add_pool(x, batch)
        out = self.readout(x)
        return out


## 6. Training utilities

Helper functions for training and evaluation. We keep track of losses and accuracies, and we use `ReduceLROnPlateau` to lower the learning rate when the validation performance stagnates.

In [ ]:
def train(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    total_correct = 0
    total_examples = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        preds = out.argmax(dim=1)
        total_correct += int((preds == batch.y).sum())
        total_examples += batch.num_graphs

    return total_loss / total_examples, total_correct / total_examples


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            loss = criterion(out, batch.y)
            total_loss += loss.item() * batch.num_graphs
            preds = out.argmax(dim=1)
            total_correct += int((preds == batch.y).sum())
            total_examples += batch.num_graphs

    return total_loss / total_examples, total_correct / total_examples


## 7. Model training

We initialize the network and launch the training loop. Intermediate metrics are logged every few epochs.

In [ ]:
model = GINNet(
    in_channels=dataset.num_features if dataset.num_features > 0 else 1,
    out_channels=dataset.num_classes,
    hidden_channels=config.hidden_channels,
    num_layers=config.num_layers,
    dropout=config.dropout,
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=config.lr_patience, factor=0.5, verbose=True)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

for epoch in range(1, config.max_epochs + 1):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print('Training complete!')


## 8. Evaluation on the test set

Once the model is trained, we report its performance on the held-out test split.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')


## 9. Learning curves

Plot the evolution of the loss and accuracy metrics to inspect the training dynamics.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss vs. Epochs')
axes[0].legend()

axes[1].plot(history['train_acc'], label='Train Accuracy')
axes[1].plot(history['val_acc'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs. Epochs')
axes[1].legend()

plt.tight_layout()
plt.show()


## 10. Making predictions on individual graphs

Finally, here is a helper function to inspect predictions on a few random test graphs. This can be extended to visualize the graphs or to perform further analysis.

In [ ]:
def predict_samples(model, dataset, num_samples=5):
    model.eval()
    indices = torch.randperm(len(dataset))[:num_samples]
    with torch.no_grad():
        for idx in indices:
            data = dataset[idx].to(device)
            logits = model(data)
            pred = logits.argmax(dim=1).item()
            print(f'Graph #{idx} | True label: {data.y.item()} | Predicted: {pred} | Nodes: {data.num_nodes}')

predict_samples(model, test_dataset)
